# Warm-up 2: Minimal Keras Sequence Classifier

Этот notebook связан с guide [Sequence Shapes and Metrics](../guides/01_sequence_shapes_and_metrics.md).

Цель:
- увидеть минимальный pipeline `tokens -> Embedding -> recurrent layer -> one label`;
- сравнить `SimpleRNN` и `GRU` без `TODO` и без большого датасета;
- посмотреть на одну и ту же synthetic classification задачу через два recurrent блока.


In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(7)
np.random.seed(7)
random.seed(7)


2026-03-24 10:58:14.557812: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 10:58:14.558201: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:58:14.560174: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:58:14.565331: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774339094.574203  189026 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774339094.57

## Synthetic dataset

Мы генерируем последовательности из целых токенов и считаем скрытый score по таблице token-weights.
Модель должна восстановить бинарную метку по всей последовательности.


In [2]:
vocab_size = 20
seq_len = 12
n_samples = 2600

token_scores = np.array(
    [0, -2, -1, -1, 0, 0, 1, 1, 2, 2, -2, -1, 1, 2, -2, 0, 1, 2, -1, 1],
    dtype=np.int32,
)

rng = np.random.default_rng(7)
X = rng.integers(1, vocab_size, size=(n_samples, seq_len), dtype=np.int32)
raw_scores = token_scores[X].sum(axis=1)
y = (raw_scores > 0).astype(np.float32)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

print('x_train shape:', x_train.shape)
print('x_test shape :', x_test.shape)
print('y_train shape:', y_train.shape)
print('Positive rate :', y_train.mean())
print()
print('Example sequence:', x_train[0])
print('Example label   :', int(y_train[0]))

x_train shape: (2080, 12)
x_test shape : (520, 12)
y_train shape: (2080,)
Positive rate : 0.6144231

Example sequence: [17  3  9  6 13  5 16 19 19  3 13 13]
Example label   : 1


In [3]:
def build_model(cell_name: str) -> tf.keras.Model:
    if cell_name == 'SimpleRNN':
        recurrent = tf.keras.layers.SimpleRNN(24)
    elif cell_name == 'GRU':
        recurrent = tf.keras.layers.GRU(24)
    else:
        raise ValueError(cell_name)

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(seq_len,)),
            tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=16, mask_zero=True),
            recurrent,
            tf.keras.layers.Dense(1, activation='sigmoid'),
        ]
    )
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


histories = {}
test_metrics = {}
models = {}

for name in ['SimpleRNN', 'GRU']:
    model = build_model(name)
    history = model.fit(
        x_train,
        y_train,
        validation_split=0.2,
        epochs=3,
        batch_size=64,
        verbose=0,
    )
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    histories[name] = history.history
    test_metrics[name] = {'loss': float(loss), 'accuracy': float(accuracy)}
    models[name] = model

test_metrics


E0000 00:00:1774339098.561640  189026 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774339098.564128  189026 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


2026-03-24 10:58:19.204586: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


{'SimpleRNN': {'loss': 0.29120609164237976, 'accuracy': 0.875},
 'GRU': {'loss': 0.26341909170150757, 'accuracy': 0.9403846263885498}}

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, history in histories.items():
    axes[0].plot(history['val_accuracy'], marker='o', label=name)
    axes[1].plot(history['val_loss'], marker='o', label=name)

axes[0].set_title('Validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('Validation loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


/tmp/ipykernel_189026/2567417260.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
sample_batch = x_test[:5]
print('Sample batch shape:', sample_batch.shape)
print(sample_batch)
print()

for name, model in models.items():
    probs = model.predict(sample_batch, verbose=0).ravel()
    preds = (probs >= 0.5).astype(np.int32)
    print(name)
    for index, (prob, pred, target) in enumerate(zip(probs, preds, y_test[:5]), start=1):
        print(f'  sample={index} prob={prob:.3f} pred={pred} target={int(target)}')
    print()


Sample batch shape: (5, 12)
[[12  6 13  2 10  1  8  5 19  6 16  4]
 [13  7  4  7 16  3  8 10 16  8 19  9]
 [11  5  1 16 13 13 10  6 10  6 13  4]
 [19  8  8  2 19 17 19  1 18  1  8  1]
 [19 19 14 10  1  1  2  3  6  6  6 14]]

SimpleRNN
  sample=1 prob=0.872 pred=1 target=1
  sample=2 prob=0.941 pred=1 target=1
  sample=3 prob=0.416 pred=0 target=1
  sample=4 prob=0.913 pred=1 target=1
  sample=5 prob=0.094 pred=0 target=0



GRU
  sample=1 prob=0.931 pred=1 target=1
  sample=2 prob=0.998 pred=1 target=1
  sample=3 prob=0.527 pred=1 target=1
  sample=4 prob=0.991 pred=1 target=1
  sample=5 prob=0.161 pred=0 target=0



## Что взять с собой дальше

- Для `many-to-one` recurrent-слой может вернуть один итоговый вектор на всю последовательность.
- `Embedding(mask_zero=True)` естественно готовит integer tokens к recurrent-слою.
- Даже маленькая synthetic задача уже показывает, как читать `train/validation/test` без перегруза архитектурой.
